# Phase 2: Data Cleaning & Preprocessing
## Session 1: Data Type Correction

**Goal:** Fix data type issues identified in Phase 1

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load dataset
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

Dataset loaded: 7043 rows, 21 columns


In [3]:
# Inspect current data types
print("Current Data Types:")
print("=" * 60)
print(df.dtypes)
print("\n" + "=" * 60)

Current Data Types:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object



In [4]:
# Check for type mismatches - focus on TotalCharges
print("Checking TotalCharges (should be numeric):")
print(f"Current type: {df['TotalCharges'].dtype}")
print(f"Sample values: {list(df['TotalCharges'].head())}")

# Check for empty/whitespace strings
empty_count = df['TotalCharges'].apply(lambda x: isinstance(x, str) and x.strip() == '').sum()
print(f"\nEmpty/whitespace strings: {empty_count}")

if empty_count > 0:
    print("\nRows with empty TotalCharges:")
    problematic = df[df['TotalCharges'].apply(lambda x: isinstance(x, str) and x.strip() == '')]
    print(problematic[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].head())

Checking TotalCharges (should be numeric):
Current type: str
Sample values: ['29.85', '1889.5', '108.15', '1840.75', '151.65']

Empty/whitespace strings: 11

Rows with empty TotalCharges:
      customerID  tenure  MonthlyCharges TotalCharges
488   4472-LVYGI       0           52.55             
753   3115-CZMZD       0           20.25             
936   5709-LVOEQ       0           80.85             
1082  4367-NUYAO       0           25.75             
1340  1371-DWPAZ       0           56.05             


---

## Type Conversion: TotalCharges

**Issue:** TotalCharges stored as string

**Conversion Strategy:**
- Replace empty strings with NaN (proper missing value marker)
- Convert to float64
- Preserve numeric precision for revenue calculations

In [5]:
# Convert TotalCharges: string -> float64
# Reason: Revenue column needs numeric type for calculations and modeling

print("Before conversion:")
print(f"Type: {df['TotalCharges'].dtype}")
print(f"NaN count: {df['TotalCharges'].isnull().sum()}")

# Convert using pd.to_numeric with errors='coerce'
# This converts empty strings to NaN automatically
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print("\nAfter conversion:")
print(f"Type: {df['TotalCharges'].dtype}")
print(f"NaN count: {df['TotalCharges'].isnull().sum()}")
print(f"Sample values: {list(df['TotalCharges'].head())}")

Before conversion:
Type: str
NaN count: 0

After conversion:
Type: float64
NaN count: 11
Sample values: [29.85, 1889.5, 108.15, 1840.75, 151.65]


In [6]:
# Verify conversion success
print("Conversion Verification:")
print("=" * 60)

# Check all numeric columns are correct type
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']

for col in numeric_cols:
    dtype = df[col].dtype
    is_numeric = pd.api.types.is_numeric_dtype(df[col])
    status = "OK" if is_numeric else "ERROR"
    print(f"{col:<20} {str(dtype):<15} [{status}]")

print("\n" + "=" * 60)

Conversion Verification:
tenure               int64           [OK]
MonthlyCharges       float64         [OK]
TotalCharges         float64         [OK]
SeniorCitizen        int64           [OK]



In [7]:
# Final data type summary
print("Final Data Types After Correction:")
print("=" * 60)

type_summary = df.dtypes.value_counts()
print(type_summary)

print("\nMissing Values After Correction:")
print("=" * 60)
missing = df.isnull().sum()
missing = missing[missing > 0]

if len(missing) > 0:
    print(missing)
else:
    print("No missing values found")

print("\n" + "=" * 60)

Final Data Types After Correction:
str        17
int64       2
float64     2
Name: count, dtype: int64

Missing Values After Correction:
TotalCharges    11
dtype: int64



---

## Summary

**Conversions Applied:**
- TotalCharges: string -> float64 (11 empty strings became NaN)

**Verification:**
- All numeric columns confirmed as numeric types
- Missing values now properly marked as NaN

**Next Steps:**
- Handle missing values (11 rows in TotalCharges)
- Encode categorical variables
- Create train/test split

**Status:** Data types corrected, ready for missing value handling

---

## Session 2: Missing Value Handling

**Goal:** Address the 11 missing TotalCharges values identified above

In [8]:
# Identify all missing values in dataset
print("Missing Value Summary:")
print("=" * 60)

missing_summary = df.isnull().sum()
missing_summary = missing_summary[missing_summary > 0]

if len(missing_summary) > 0:
    for col, count in missing_summary.items():
        pct = (count / len(df)) * 100
        print(f"{col:<20} {count:>5} rows ({pct:.2f}%)")
else:
    print("No missing values found")

print("\n" + "=" * 60)

Missing Value Summary:
TotalCharges            11 rows (0.16%)



In [9]:
# Investigate TotalCharges missing values
print("TotalCharges Missing Value Analysis:")
print("=" * 60)

missing_idx = df['TotalCharges'].isnull()
missing_rows = df[missing_idx]

print(f"\nTotal missing: {missing_idx.sum()} rows")
print("\nCharacteristics of missing rows:")
print(f"  - Tenure range: {missing_rows['tenure'].min()} to {missing_rows['tenure'].max()}")
print(f"  - MonthlyCharges range: ${missing_rows['MonthlyCharges'].min():.2f} to ${missing_rows['MonthlyCharges'].max():.2f}")
print(f"  - Churn rate: {(missing_rows['Churn'] == 'Yes').sum()}/{len(missing_rows)} ({(missing_rows['Churn'] == 'Yes').mean() * 100:.1f}%)")

print("\nSample rows with missing TotalCharges:")
print(missing_rows[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].head())

print("\n" + "=" * 60)

TotalCharges Missing Value Analysis:

Total missing: 11 rows

Characteristics of missing rows:
  - Tenure range: 0 to 0
  - MonthlyCharges range: $19.70 to $80.85
  - Churn rate: 0/11 (0.0%)

Sample rows with missing TotalCharges:
      customerID  tenure  MonthlyCharges  TotalCharges Churn
488   4472-LVYGI       0           52.55           NaN    No
753   3115-CZMZD       0           20.25           NaN    No
936   5709-LVOEQ       0           80.85           NaN    No
1082  4367-NUYAO       0           25.75           NaN    No
1340  1371-DWPAZ       0           56.05           NaN    No



---

## Missing Value Strategy: TotalCharges

**Why values are missing:**
- All 11 rows have `tenure = 0` (brand new customers)
- TotalCharges is cumulative revenue, which doesn't exist yet for new signups
- This is MNAR (Missing Not At Random) - systematic, not random

**Options:**
1. Drop rows: Loses 11 customers (0.16% of data) - acceptable but unnecessary
2. Impute with 0: Logical but creates outliers (everyone else has positive values)
3. Impute with MonthlyCharges: Assumes first month was billed - most realistic

**Decision: Impute with MonthlyCharges**
- Reason: Reflects expected first bill for tenure=0 customers
- Safe because: MonthlyCharges already reflects their service level
- Trade-off: Assumes billing happened (might not be true for all 11)

In [10]:
# Handle missing TotalCharges values
# Strategy: Impute with MonthlyCharges for tenure=0 customers
# Reasoning: New customers haven't accumulated charges yet, but MonthlyCharges reflects expected first bill

print("Before Imputation:")
print(f"Missing TotalCharges: {df['TotalCharges'].isnull().sum()}")

# Create boolean mask for missing values
missing_mask = df['TotalCharges'].isnull()

# Impute: fill missing TotalCharges with MonthlyCharges
df.loc[missing_mask, 'TotalCharges'] = df.loc[missing_mask, 'MonthlyCharges']

print("\nAfter Imputation:")
print(f"Missing TotalCharges: {df['TotalCharges'].isnull().sum()}")

# Verify imputation worked correctly
print("\nImputed values (should match MonthlyCharges for tenure=0):")
imputed_customers = df[df['tenure'] == 0][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].head(5)
print(imputed_customers)

Before Imputation:
Missing TotalCharges: 11

After Imputation:
Missing TotalCharges: 0

Imputed values (should match MonthlyCharges for tenure=0):
      customerID  tenure  MonthlyCharges  TotalCharges
488   4472-LVYGI       0           52.55         52.55
753   3115-CZMZD       0           20.25         20.25
936   5709-LVOEQ       0           80.85         80.85
1082  4367-NUYAO       0           25.75         25.75
1340  1371-DWPAZ       0           56.05         56.05


In [11]:
# Final verification: check entire dataset for missing values
print("Final Missing Value Check:")
print("=" * 60)

final_missing = df.isnull().sum()
final_missing_total = final_missing.sum()

if final_missing_total == 0:
    print("✓ No missing values in dataset")
else:
    print("⚠ Missing values still present:")
    print(final_missing[final_missing > 0])

print(f"\nDataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\n" + "=" * 60)

Final Missing Value Check:
✓ No missing values in dataset

Dataset shape: 7043 rows × 21 columns



---

## Session 2 Summary

**Missing Values Handled:**
- TotalCharges: 11 rows (0.16%) imputed with MonthlyCharges
- Reasoning: All missing rows had tenure=0 (new customers with no billing history yet)
- No rows dropped - preserved full dataset

**Information Trade-offs:**
- Lost: Exact billing status of 11 new customers (may not have been billed yet)
- Gained: Complete dataset without nulls, realistic revenue estimates for new signups

**Validation:**
- Zero missing values remaining
- Imputed values logical (TotalCharges = MonthlyCharges for tenure=0)
- Dataset integrity maintained (7,043 rows preserved)

**Status:** Ready for categorical encoding (Session 3)

---

## Session 3: Target & Binary Encoding

**Goal:** Encode Churn target and all binary Yes/No features

In [12]:
# Identify all Yes/No binary columns
print("Binary Column Detection:")
print("=" * 60)

# Check unique values for each object column
binary_cols = []
for col in df.select_dtypes(include='object').columns:
    unique_vals = df[col].unique()
    if set(unique_vals) == {'Yes', 'No'}:
        binary_cols.append(col)
        print(f"{col:<20} → ['Yes', 'No']")

print(f"\nTotal binary columns found: {len(binary_cols)}")
print("\n" + "=" * 60)

Binary Column Detection:
Partner              → ['Yes', 'No']
Dependents           → ['Yes', 'No']
PhoneService         → ['Yes', 'No']
PaperlessBilling     → ['Yes', 'No']
Churn                → ['Yes', 'No']

Total binary columns found: 5



/var/folders/7f/mldk3sgj4550r3kh5dv86x140000gn/T/ipykernel_75932/109833199.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


---

## Encoding Strategy

**Target Variable (Churn):**
- Create `churn_flag`: 0 = No (retained), 1 = Yes (churned)
- Separate from features for clarity
- Keep original column for reference

**Binary Features (Yes/No):**
- Encode consistently: Yes → 1, No → 0
- Rename with `_flag` suffix for explicitness
- Keep original columns temporarily for validation

In [13]:
# Encode target variable: Churn → churn_flag
# Reasoning: Separate target from features, clear binary encoding for modeling

print("Target Encoding:")
print("=" * 60)

# Create binary target: 1 = churned, 0 = retained
df['churn_flag'] = (df['Churn'] == 'Yes').astype(int)

print(f"Original Churn distribution:")
print(df['Churn'].value_counts())
print(f"\nEncoded churn_flag distribution:")
print(df['churn_flag'].value_counts())

# Verify encoding is correct
print("\nEncoding verification:")
print(df[['Churn', 'churn_flag']].drop_duplicates().sort_values('churn_flag'))

print("\n" + "=" * 60)

Target Encoding:
Original Churn distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Encoded churn_flag distribution:
churn_flag
0    5174
1    1869
Name: count, dtype: int64

Encoding verification:
  Churn  churn_flag
0    No           0
2   Yes           1



In [14]:
# Encode all binary Yes/No features
# Reasoning: Consistent encoding (Yes=1, No=0) for all binary categorical features
# Exclude 'Churn' since we already created churn_flag as the target

print("Binary Feature Encoding:")
print("=" * 60)

# Get binary columns excluding Churn (already handled)
binary_features = [col for col in binary_cols if col != 'Churn']

print(f"Encoding {len(binary_features)} binary features:")
for col in binary_features:
    new_col_name = col.lower().replace(' ', '_') + '_flag'
    df[new_col_name] = (df[col] == 'Yes').astype(int)
    print(f"  {col:<20} → {new_col_name}")

print("\n" + "=" * 60)

Binary Feature Encoding:
Encoding 4 binary features:
  Partner              → partner_flag
  Dependents           → dependents_flag
  PhoneService         → phoneservice_flag
  PaperlessBilling     → paperlessbilling_flag



In [15]:
# Verify binary encoding correctness
print("Encoding Verification:")
print("=" * 60)

# Check a few binary features to ensure encoding is correct
sample_features = binary_features[:3] if len(binary_features) >= 3 else binary_features

for col in sample_features:
    flag_col = col.lower().replace(' ', '_') + '_flag'
    print(f"\n{col} → {flag_col}:")
    print(df[[col, flag_col]].drop_duplicates().sort_values(flag_col))

print("\n" + "=" * 60)

Encoding Verification:

Partner → partner_flag:
  Partner  partner_flag
1      No             0
0     Yes             1

Dependents → dependents_flag:
  Dependents  dependents_flag
0         No                0
6        Yes                1

PhoneService → phoneservice_flag:
  PhoneService  phoneservice_flag
0           No                  0
1          Yes                  1



In [16]:
# Check current dataset structure
print("Dataset Structure After Binary Encoding:")
print("=" * 60)

print(f"\nShape: {df.shape[0]} rows × {df.shape[1]} columns")

# Count encoded features
encoded_features = [col for col in df.columns if col.endswith('_flag')]
print(f"\nEncoded binary features: {len(encoded_features)}")
print(f"  - Target: churn_flag")
print(f"  - Features: {len(encoded_features) - 1}")

# Show data types
print(f"\nData type summary:")
print(df.dtypes.value_counts())

print("\n" + "=" * 60)

Dataset Structure After Binary Encoding:

Shape: 7043 rows × 26 columns

Encoded binary features: 5
  - Target: churn_flag
  - Features: 4

Data type summary:
str        17
int64       7
float64     2
Name: count, dtype: int64



---

## Session 3 Summary

**Target Variable:**
- Created `churn_flag`: 0=retained, 1=churned
- Preserved original `Churn` column for reference
- Encoding: Yes → 1, No → 0

**Binary Features Encoded:**
- All Yes/No features converted to `*_flag` columns
- Consistent encoding logic across all binary variables
- Original columns retained for validation

**Encoding Logic (Reversible):**
```python
# Decoding logic if needed:
# 1 → 'Yes', 0 → 'No'
```

**Next Steps:**
- Drop original string columns (after validation)
- Encode multi-class categorical features
- Create final feature set for modeling

**Status:** Binary encoding complete, ready for multi-class encoding